# if문에 대한 연구와 사용법
디자인 패턴, 가독성 향상, 성능 향상 등을 위한 코딩에 대한 이야기입니다.

# if문의 비용

`if`문 자체는 `if` 다음의 연산을 수행하여 결과(`boolean`)에 따라 다음 코드 수행 위치를 이동시키는 것이 전부입니다.

여기에서 `if`문 뒤에 수행하는 연산을 제외하면 `if`문 자체에 대한 오버헤드는 거의 없다고 볼 수 있습니다.

In [ ]:
import timeit

# 반복 횟수 
N = 10_000_000

# 기준이 되는 pass에 드는 시간
baseline = timeit.timeit(
    "pass",
    number=N
)

# x = True인 경우에 대한 1000만번 실행 시간
if_true_time = timeit.timeit(
    """
if x:
    pass
""",
    setup="x = True",
    number=N
)

# x = False에 대한 1000만번 실행 시간
if_false_time = timeit.timeit(
    """
if x:
    pass
""",
    setup="x = False",
    number=N
)

print("pass만 실행:", baseline)
print("if True 실행:", if_true_time)
print("if False 실행:", if_false_time)

# 한 번 실행에 대한 시간 -> (초 단위) -> ns값으로 변환하기 위해 10억을 곱하기
print("if True 추가 비용(ns):", (if_true_time - baseline) / N * 1_000_000_000)
print("if False 추가 비용(ns):", (if_false_time - baseline) / N * 1_000_000_000)

pass만 실행(ms): 0.07675649994052947
if True 실행: 0.10124069999437779
if False 실행: 0.08183529996313155
if True 추가 비용(ns): 2.4484200053848326
if False 추가 비용(ns): 0.5078800022602081


# `if`문을 잘 쓰기 위한 방법들

1. **조건식 비용 관리**: 비싼 조건은 필요할 때에만 실행되도록 뒤쪽에 배치해야 합니다.
2. **단락 평가의 최적화**(`short-circuit`): `and`같이 모두 만족해야 하는 경우, 파이썬은 앞부분만 연산한 뒤 `False`인 경우에는 바로 탈락시키기 때문에 앞부분에 탈락 확률이 높은 변수를 두는 것이 좋습니다.
3. **조건 순서 최적화**: 위와 비슷하게 빠르게 성공시키거나, 계산 비용이 적은 조건을 먼저 나오게 하면 좋습니다.
4. **`if`/`elif`**: 배타적인 조건이면 불필요한 검사 방지를 위해 `elif`/`else`를 사용하면 좋습니다.
5. **내부 연산의 최적화**: 내부 `if` 연산은 `O(1)`에 가까운 것을 사용하는 것이 일반적으로 좋습니다.
6. **딕셔너리 분기처리**: 명령어에 따라 함수를 실행하는 경우에는 **dispatch table(전략 패턴)**을 이용하기 적합합니다.
7. **분기 보호**: 가독성과 유지보수를 위해 `guard clause` 방식을 사용하면 좋습니다. (중첩 `if`문 자제)
8. **`match-case`의 위치와 성능감각**: `match-case`는 성능보다는 구조분해와 패턴 매칭에 읽기 좋게 쓰입니다. 단순 값 비교 시에는 비슷한 성능을 보이고, 복잡한 매칭에서는 무거울 수 있습니다.

In [ ]:
# 예를 들어 보겠습니다.
import random

def getBool(true_prob: int):
  return random.random() < true_prob / 100

class User:
  def __init__(self, is_login: bool, is_active: bool):
    self.is_login = is_login
    self.is_active = is_active

# f1 = 70% 확률로 False인 객체
f1 = getBool(30)

# t1 = 80% 확률로 True 객체
t1 = getBool(80)

# 위와같은 변수의 경우

# 배타적 조건
# 높은 확률 먼저 계산
def example_low_calc(t1: bool, f1: bool) -> None:
  if f1 and t1:
    print("both True")
  # 배타적 조건으로 elif, True일 확률이 높은 t1 먼저 (t1은 80%, f1은 30%)
  elif t1:
    print("t1 only True")
  elif f1:
    print("f1 only True")
  else:
    print("both not True")
  
  # 아래와 같이도 가능
  match (t1, f1):
    case True, True:
      print("both True")
    case True, _:
      print("t1 only True")
    case _, True:
      print("f1 only True")
    case _:
      print("both not True")

example_low_calc(t1, f1)

# guard clause
def example_guard_clause():  
  # 70% 확률로 존재하는 사용자,
  # 40%확률로 로그인
  # login되어있으면 30%확률로 활성화
  user = User(is_login := getBool(40), is_active=(getBool(30) and is_login)) if getBool(70) else None

  # 잘못된 예시
  if user is not None:
    if user.is_login:
      if user.is_active:
        print("user_active")
        # 너무 depth가 깊어짐


  # 올바른 guard clause의 사용
  if user is None:
    print("no user")
    return
  print("user exist")

  if not user.is_login:
    print("user not login")
    return
  print("user logged in")
  
  if not user.is_active:
    print("user not active")
    return
  print("user active")

def example_faster_search():
  a = [i for i in range(1, 101)]
  
  # 느린 연산 
  if random.randint(1, 201) in a:
    print("exist")
  
  # 반복 조회에서는 빠른 연산
  s = set(a)
  if random.randint(1, 201) in s:
    print("exist")

# match-case가 비효율적인 경우
def example_inefficient_match():
  # match-case가 오래 걸리는 경우는 단순 값 비교가 아닌 직접 뜯어서 찾아봐야 하는 경우입니다.
  data = {
    "type": "IT",
    "name": "developer",
    "salary": 3000
  }

  # 이전에는 if를 사용하였지만 아래와 같은 경우에는 비효율적입니다.
  
  # 1. dict 타입의 내부 값을 하나씩 확인해야 하는 경우
  match data:
    case {"type": "IT", "name": "frontend", "salary": _}:
      print("IT frontend ?")
    case {"type": "IT", "name": "developer", "salary": 3000}:
      print("IT developer 3000")
  
  # 위와 같은 경우에는 이렇게 직접 조건을 확인해 속도를 더 향상할 수 있습니다.
  if data.get("type") == "IT" and data.get("name") == "frontend" and "salary" in data:
    print("IT frontend ?")
  elif data.get("type") == "IT" and data.get("name") == "developer" and data.get("salary") == 3000:
    print("IT developer 3000")
    

f1 only True
f1 only True
